# Multiple testing in A/B/n

A checkout experiment has a control and two variants. Compare a numeric
outcome and a categorical outcome with Holm and Benjamini–Hochberg, then
use the pandas quantile method to ask a different question about a numeric
outcome.

- [Build the comparison family](#comparison-family)
- [Compare Holm and Benjamini–Hochberg](#adjusted-p-values)
- [Use the quantile method to identify a winner](#quantile-winner)

<a id="comparison-family"></a>

## 1. Build the comparison family

Each variant is compared with control on revenue and order status, giving
four comparisons. HypEx adjusts p-values separately by test type,
so the two revenue comparisons form one family and the two status
comparisons form another. The
`manual_review` status exists only in variant 2. A zero observed count in
one arm is compatible with a valid chi-square comparison when expected
counts are adequate. The status test compares the whole distribution.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from hypex import ABTest
from hypex.analyzers.ab import ABAnalyzer
from hypex.dataset import Dataset, TargetRole, TreatmentRole
from hypex.utils import ABNTestMethodsEnum

# Synthetic data preparation (NumPy and pandas).
rng = np.random.default_rng(101)
n_per_arm = 600
status_counts = [
    {"paid": 360, "abandoned": 240},
    {"paid": 364, "abandoned": 236},
    {"paid": 355, "abandoned": 205, "manual_review": 40},
]
frame = pd.concat(
    [
        pd.DataFrame(
            {
                "variant": variant,
                "revenue": rng.normal(40 + lift, 12, n_per_arm),
                "status": rng.permutation(np.repeat(list(counts), list(counts.values()))),
            }
        )
        for variant, (lift, counts) in enumerate(zip([0, 1.2, 4], status_counts))
    ],
    ignore_index=True,
)
# Local data check (pandas). Preview two rows per arm and check synthetic status counts.
display(frame.groupby("variant").head(2).reset_index(drop=True))

counts_by_arm = pd.crosstab(frame["variant"], frame["status"])
counts_by_arm

,variant,revenue,status
0,0,30.518170,paid
1,0,15.584494,abandoned
2,1,47.175588,paid
3,1,30.450732,paid
4,2,33.858128,paid
5,2,57.715403,paid


status,abandoned,manual_review,paid
variant,,,
0,240,0,360
1,236,0,364
2,205,40,355


<a id="adjusted-p-values"></a>

## 2. Compare adjusted p-values

Holm controls the probability of at least one false rejection in the family.
Benjamini–Hochberg (`fdr_bh`) controls the expected fraction of false
rejections among discoveries under independence or suitable positive
dependence. Shared control and multiple outcomes can introduce dependence.
Choose the method according to the decision and its assumptions before
examining the results.

In [2]:
roles = {
    "variant": TreatmentRole(int),
    "revenue": TargetRole(float),
    "status": TargetRole(str),
}
# HypEx input boundary. For Holm or BH on Spark, construct this Dataset from Spark data.
data = Dataset(data=frame, roles=roles)

holm = ABTest(
    additional_tests=["t-test", "chi2-test"],
    equal_variance=False,
    multitest_method=ABNTestMethodsEnum.holm,
).execute(data)
print("Holm")
display(holm.multitest)

bh = ABTest(
    additional_tests=["t-test", "chi2-test"],
    equal_variance=False,
    multitest_method=ABNTestMethodsEnum.fdr_bh,
).execute(data)
print("Benjamini–Hochberg")
display(bh.multitest)

Holm


,field,test,old p-value,new p-value,correction,rejected,group
0,revenue,TTest,0.01197821929093111,0.01197821929093111,1.0,True,1
1,revenue,TTest,5.353867110602919e-12,1.0707734221205838e-11,0.5,True,2
2,status,Chi2Test,0.9334927366713109,0.9334927366713109,1.0,False,1
3,status,Chi2Test,2.2613951798738886e-05,4.522790359747777e-05,0.5,True,2


Benjamini–Hochberg


,field,test,old p-value,new p-value,correction,rejected,group
0,revenue,TTest,0.01197821929093111,0.01197821929093111,1.0,True,1
1,revenue,TTest,5.353867110602919e-12,1.0707734221205838e-11,0.5,True,2
2,status,Chi2Test,0.9334927366713109,0.9334927366713109,1.0,False,1
3,status,Chi2Test,2.2613951798738886e-05,4.522790359747777e-05,0.5,True,2


In [3]:
# Local check (pandas and SciPy) on a two-row count table.
# For equally sized arms, the pandas HypEx chi-square path multiplies each
# arm's counts by 0.5. Keep the zero in control for manual_review.
observed = counts_by_arm.loc[[0, 2]]
weighted_test = chi2_contingency(observed * 0.5)
reported = next(
    row["old p-value"] for row in holm.multitest.to_records()
    if row["test"] == "Chi2Test" and str(row["group"]) == "2"
)
# Agreement confirms the HypEx comparison uses the weighted table with the one-arm category.
np.testing.assert_allclose(float(reported), weighted_test.pvalue, rtol=1e-7, atol=0)

pd.Series(
    {
        "smallest expected weighted count": weighted_test.expected_freq.min(),
        "HypEx p-value before multiple testing correction": float(reported),
    }
)

smallest expected weighted count                    10.000000
HypEx p-value before multiple testing correction     0.000023
dtype: float64

The check confirms that the comparison retains a category even when one arm
has zero observations for it. The calculation matches the weighted
table used by HypEx on pandas. A chi-square test on the raw counts
uses a different statistic. An adjusted rejection for status indicates a
change somewhere in the status distribution. The observed counts show how
the statuses differ.

<a id="quantile-winner"></a>

## 3. Use the quantile method for a numeric winner

On pandas, `multitest_method="quantile"` checks whether one arm's numeric
mean is sufficiently higher than every other arm. It estimates critical
values by simulation. This example has equal group sizes and generates
revenue with the same variance in every arm. Set the analyzer seed to make
the simulation reproducible.

The report contains an `accepted hypothesis` per numeric target. Zero means
no winner was identified. A positive value is the one-based position of the
winning arm in group order, including control. It differs from the
control-versus-variant adjusted p-value table above.

In [4]:
numeric_data = data.select(["variant", "revenue"])
quantile_test = ABTest(multitest_method="quantile")
quantile_test.experiment.set_params({ABAnalyzer: {"random_state": 101}})
quantile_report = quantile_test.execute(numeric_data).multitest

group_order = sorted(numeric_data.select("variant").unique()["variant"])
position = int(quantile_report.to_records()[0]["accepted hypothesis"])
winner = group_order[position - 1] if position else None
display(numeric_data.groupby("variant").agg({"revenue": "mean"}))
display(quantile_report)
print("No winner identified." if winner is None else f"Winning variant: {winner}")

,revenue
variant,
0,39.550954
1,41.306119
2,44.293252


,field,accepted hypothesis
0,revenue,3


Winning variant: 2
